# LIAR2 FDHN ensemble experiment

This experiment combines the current feature/model changes: fine-tuned **DistilBERT** on justification, engineered speaker-history features, and no statement branch. It additionally trains a **3-seed ensemble** and averages softmax probabilities, uses **0.1 label smoothing**, and adds a small ordinal penalty so predictions farther from the correct ordered truth label cost more.

Each member checkpoint is saved separately in `liar2/` using a distinct seed. Training selects each member by validation accuracy and applies early stopping. The base paper-reference path remains available when the environment flags are unset.

**Requirements:** PyTorch, Transformers, scikit-learn, and access to the pretrained DistilBERT files. Three fine-tuning runs can be very compute-intensive; a CUDA GPU is recommended. These are experiments, not guaranteed improvements. Compare validation metrics before relying on the held-out test result.

Run the code cell below from any directory inside this repository.

In [ ]:
import os
import runpy
from pathlib import Path

script_name = Path("segmentation_experiments/5.1.LIAR2.py")
repo_root = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / script_name).is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate {script_name} from {Path.cwd()}")

data_dir = repo_root / "liar2"
missing_splits = [name for name in ("train.csv", "valid.csv", "test.csv")
                  if not (data_dir / name).is_file()]
if missing_splits:
    raise FileNotFoundError(f"Missing LIAR2 split(s) in {data_dir}: {missing_splits}")

previous_cwd = Path.cwd()
previous_mode = os.environ.get("LIAR2_PRETRAINED_JUSTIFICATION")
previous_numeric_mode = os.environ.get("LIAR2_ENGINEERED_NUMERICAL_FEATURES")
previous_statement_mode = os.environ.get("LIAR2_USE_STATEMENT_BRANCH")
previous_smoothing = os.environ.get("LIAR2_LABEL_SMOOTHING")
previous_ordinal_weight = os.environ.get("LIAR2_ORDINAL_LOSS_WEIGHT")
previous_ensemble_size = os.environ.get("LIAR2_ENSEMBLE_SIZE")
try:
    os.chdir(data_dir)
    os.environ["LIAR2_PRETRAINED_JUSTIFICATION"] = "1"
    os.environ["LIAR2_ENGINEERED_NUMERICAL_FEATURES"] = "1"
    os.environ["LIAR2_USE_STATEMENT_BRANCH"] = "0"
    os.environ["LIAR2_LABEL_SMOOTHING"] = "0.1"
    os.environ["LIAR2_ORDINAL_LOSS_WEIGHT"] = "0.1"
    os.environ["LIAR2_ENSEMBLE_SIZE"] = "3"
    runpy.run_path(str(repo_root / script_name), run_name="__main__")
finally:
    os.chdir(previous_cwd)
    if previous_mode is None:
        os.environ.pop("LIAR2_PRETRAINED_JUSTIFICATION", None)
    else:
        os.environ["LIAR2_PRETRAINED_JUSTIFICATION"] = previous_mode
    if previous_numeric_mode is None:
        os.environ.pop("LIAR2_ENGINEERED_NUMERICAL_FEATURES", None)
    else:
        os.environ["LIAR2_ENGINEERED_NUMERICAL_FEATURES"] = previous_numeric_mode
    if previous_statement_mode is None:
        os.environ.pop("LIAR2_USE_STATEMENT_BRANCH", None)
    else:
        os.environ["LIAR2_USE_STATEMENT_BRANCH"] = previous_statement_mode
    for key, previous_value in (("LIAR2_LABEL_SMOOTHING", previous_smoothing),
                                ("LIAR2_ORDINAL_LOSS_WEIGHT", previous_ordinal_weight),
                                ("LIAR2_ENSEMBLE_SIZE", previous_ensemble_size)):
        if previous_value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = previous_value
